# Model Evaluation and Threshold Selection

높은 점수 하나만 확인하지 않고, **baseline 비교 → validation 평가 → 정책 기반 임계값 선택 → 봉인된 test 평가** 순서로 모델을 검증합니다.

## 실험 질문

1. 회귀 모델은 train 평균을 예측하는 baseline보다 나은가?
2. 악성 종양을 놓치는 비용이 클 때 Accuracy 외에 어떤 지표를 봐야 하는가?
3. Recall 정책을 만족하는 임계값을 test를 보지 않고 선택할 수 있는가?

> scikit-learn 예제 데이터를 사용한 학습 목적의 실험이며 실제 의료 판단에 사용할 수 없습니다.


## 1. Environment and reproducibility

- Python, NumPy, pandas, scikit-learn
- `random_state=42`
- train / validation / test = 60% / 20% / 20%
- 전처리 통계와 모델은 train에서만 학습
- validation은 모델·임계값 선택에 사용
- test는 모든 선택을 고정한 뒤 마지막에 한 번만 평가


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
pd.options.display.float_format = "{:.4f}".format


## 2. Data split and leakage checks


In [2]:
def assert_disjoint(*frames):
    """원본 행 인덱스를 기준으로 분할 간 중복이 없는지 검사합니다."""
    index_sets = [set(frame.index) for frame in frames]
    for left in range(len(index_sets)):
        for right in range(left + 1, len(index_sets)):
            assert index_sets[left].isdisjoint(index_sets[right])


diabetes = load_diabetes(as_frame=True)
X_reg, y_reg = diabetes.data, diabetes.target
X_reg_train, X_reg_temp, y_reg_train, y_reg_temp = train_test_split(
    X_reg, y_reg, test_size=0.4, random_state=SEED
)
X_reg_valid, X_reg_test, y_reg_valid, y_reg_test = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.5, random_state=SEED
)
assert_disjoint(X_reg_train, X_reg_valid, X_reg_test)

breast = load_breast_cancer(as_frame=True)
X_cls = breast.data
# 원본 target의 malignant=0을 탐지 대상 positive class 1로 변환합니다.
y_cls = (breast.target == 0).astype(int)
X_cls_train, X_cls_temp, y_cls_train, y_cls_temp = train_test_split(
    X_cls, y_cls, test_size=0.4, stratify=y_cls, random_state=SEED
)
X_cls_valid, X_cls_test, y_cls_valid, y_cls_test = train_test_split(
    X_cls_temp, y_cls_temp, test_size=0.5, stratify=y_cls_temp, random_state=SEED
)
assert_disjoint(X_cls_train, X_cls_valid, X_cls_test)

split_summary = pd.DataFrame(
    [
        {"task": "regression", "train": len(X_reg_train), "validation": len(X_reg_valid), "test": len(X_reg_test)},
        {"task": "classification", "train": len(X_cls_train), "validation": len(X_cls_valid), "test": len(X_cls_test)},
    ]
)
print(split_summary.to_string(index=False))
print(f"classification positive rate: {y_cls.mean():.4f}")


          task  train  validation  test
    regression    265          88    89
classification    341         114   114
classification positive rate: 0.3726


## 3. Regression: candidate vs train-mean baseline

RMSE 자체의 크기만 보는 대신, train target 평균을 반복 예측하는 baseline과 같은 validation set에서 비교합니다.


In [3]:
def build_regression_report(X_train, y_train, X_valid, y_valid):
    model = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]
    )
    model.fit(X_train, y_train)

    candidates = {
        "linear_regression": model.predict(X_valid),
        "train_mean_baseline": np.full(len(y_valid), y_train.mean()),
    }
    rows = []
    for name, prediction in candidates.items():
        rows.append(
            {
                "candidate": name,
                "MAE": mean_absolute_error(y_valid, prediction),
                "RMSE": mean_squared_error(y_valid, prediction) ** 0.5,
                "R2": r2_score(y_valid, prediction),
            }
        )

    report = pd.DataFrame(rows)
    model_rmse = report.loc[report["candidate"] == "linear_regression", "RMSE"].iloc[0]
    baseline_rmse = report.loc[report["candidate"] == "train_mean_baseline", "RMSE"].iloc[0]
    assert model_rmse < baseline_rmse
    return model, report


regression_model, regression_report = build_regression_report(
    X_reg_train, y_reg_train, X_reg_valid, y_reg_valid
)
print(regression_report.to_string(index=False))


          candidate     MAE    RMSE      R2
  linear_regression 38.2213 49.1497  0.5810
train_mean_baseline 67.5162 76.1614 -0.0062


### Interpretation

Linear Regression의 validation RMSE는 baseline보다 약 **27.01** 낮습니다. R²는 약 **0.581**로, baseline보다 설명력이 분명히 개선됐습니다. 모델 성능은 단독 점수가 아니라 동일한 조건의 기준 모델과 비교해야 의미가 있습니다.


## 4. Classification: threshold metrics and ranking metrics

악성을 positive class로 정의합니다. 임계값 0.5의 Accuracy, Precision, Recall, F1과 확률 순위를 평가하는 ROC-AUC, AP를 함께 확인합니다.


In [4]:
def build_classification_report(X_train, y_train, X_valid, y_valid):
    model = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
        ]
    )
    model.fit(X_train, y_train)

    probability = model.predict_proba(X_valid)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    report = pd.Series(
        {
            "accuracy": accuracy_score(y_valid, prediction),
            "precision": precision_score(y_valid, prediction, zero_division=0),
            "recall": recall_score(y_valid, prediction, zero_division=0),
            "f1": f1_score(y_valid, prediction, zero_division=0),
            "roc_auc": roc_auc_score(y_valid, probability),
            "AP": average_precision_score(y_valid, probability),
        }
    )
    assert report.between(0.0, 1.0).all()
    return model, probability, report


classifier, valid_probability, classification_report = build_classification_report(
    X_cls_train, y_cls_train, X_cls_valid, y_cls_valid
)
print(classification_report.to_string())


accuracy    0.9737
precision   1.0000
recall      0.9302
f1          0.9639
roc_auc     1.0000
AP          1.0000


### Interpretation

임계값 0.5의 Recall은 약 **0.9302**로 일부 악성 사례를 놓칩니다. Accuracy가 높더라도 False Negative의 비용이 큰 문제에서는 Recall을 반드시 확인해야 합니다. ROC-AUC와 AP가 높다는 사실도 특정 임계값에서 모든 악성을 찾았다는 의미는 아닙니다.


## 5. Threshold policy and sealed test evaluation

운영 정책을 **validation Recall ≥ 0.90인 후보 중 F1 최대화**로 정의합니다. 동률이면 Precision과 threshold가 높은 후보를 선택합니다.


In [5]:
def choose_threshold(y_true, probability, minimum_recall=0.90):
    rows = []
    for threshold in np.linspace(0.05, 0.95, 91):
        prediction = (probability >= threshold).astype(int)
        rows.append(
            {
                "threshold": threshold,
                "precision": precision_score(y_true, prediction, zero_division=0),
                "recall": recall_score(y_true, prediction, zero_division=0),
                "f1": f1_score(y_true, prediction, zero_division=0),
            }
        )

    table = pd.DataFrame(rows)
    eligible = table.loc[table["recall"] >= minimum_recall].copy()
    if eligible.empty:
        raise RuntimeError("Recall 정책을 만족하는 임계값이 없습니다.")

    selection = eligible.sort_values(
        ["f1", "precision", "threshold"], ascending=[False, False, False]
    ).iloc[0]
    return selection, table


def evaluate_at_threshold(model, X_test, y_test, threshold):
    probability = model.predict_proba(X_test)[:, 1]
    prediction = (probability >= threshold).astype(int)
    return pd.Series(
        {
            "AP": average_precision_score(y_test, probability),
            "precision": precision_score(y_test, prediction, zero_division=0),
            "recall": recall_score(y_test, prediction, zero_division=0),
            "f1": f1_score(y_test, prediction, zero_division=0),
        }
    )


selection, threshold_table = choose_threshold(
    y_cls_valid, valid_probability, minimum_recall=0.90
)
test_report = evaluate_at_threshold(
    classifier, X_cls_test, y_cls_test, float(selection["threshold"])
)

print("[validation policy]")
print("recall >= 0.90, then maximize F1 / precision / threshold")
print("\n[sealed test result]")
print(test_report.to_string())


[validation policy]
recall >= 0.90, then maximize F1 / precision / threshold

[sealed test result]
AP          0.9920
precision   0.9756
recall      0.9524
f1          0.9639


## Conclusion

- 회귀 모델은 train-mean baseline보다 낮은 MAE와 RMSE를 기록했습니다.
- 분류에서는 Accuracy만 보지 않고, 악성을 놓치는 비용을 반영해 Recall을 함께 평가했습니다.
- 임계값은 validation에서 정책에 따라 선택하고 fitted model과 함께 고정했습니다.
- test 결과는 다시 모델이나 임계값을 고르는 데 사용하지 않았습니다.

이 실험에서 가장 중요한 결과는 높은 점수 자체보다 **어떤 데이터를 어떤 결정에 사용했는지 설명할 수 있는 평가 절차**입니다.
